# Late Fusion: CNN (image) + Random Forest (metadata)

**Input.** Combined `fusion_df` has `image_name, patient_id, target, p_cnn, p_rf`. 
These are the base models' predictions on the validation set (1,235 images, 25 melanomas, 85 patients). 
The fusion models are **trained on this table only**. 
`fusion_test_df` has the same columns for the test set and is used once, at the end.

**Fusion methods**

| # | Method | Formula | Learned parameters |
|---|---|---|---|
| 1 | Simple average | $(p_{cnn} + p_{rf})/2$ | 0 |
| 2 | Rank average | mean of each score's percentile within the *validation* score distribution | 0 |
| 3 | Weighted logit average | $w\cdot\text{logit}(p_{cnn}) + (1-w)\cdot\text{logit}(p_{rf})$ | 1 ($w$) |
| 4 | Logistic-regression stacking | $\sigma(a + b\cdot\text{logit}(p_{cnn}) + c\cdot\text{logit}(p_{rf}))$ | 3 |

**Evaluation plan**
1. **Model selection on validation.** Use repeated, patient-grouped, stratified 5-fold CV on `fusion_df`.
   Each fold's fusion model is fit on 4/5 of the patients and scored on the other 1/5. That gives honest
   out-of-fold (OOF) predictions for every method, plus CNN-only and RF-only as reference points.
2. **Thresholds.** Each fold picks its decision threshold on its *training* part (the highest threshold
   that keeps sensitivity ≥ `TARGET_SENS`) and applies it to the held-out part. Sensitivity and
   specificity are therefore out-of-sample too.
3. **Choose the winner on validation CV (primary metric: PR-AUC).** Refit every method on the full
   `fusion_df` and report all of them on the test set once. The winner is fixed *before* looking at test,
   so the test numbers stay unbiased.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix)

SEED        = 42
N_SPLITS    = 5      # ~5 melanomas per held-out fold
N_REPEATS   = 20     # repeat CV with different shuffles: one split of 25 positives is too noisy
TARGET_SENS = 0.80   # operating point: catch at least 80% of melanomas
N_BOOT      = 2000   # bootstrap resamples for test-set confidence intervals
EPS         = 1e-4   # clip probabilities before logit (RF can output exactly 0 or 1)

# Plot colours: base models in grey, the four fusion methods in fixed categorical order
COLORS = {"CNN only": "#8a8984", "RF only": "#c3c2b7",
          "1. Simple avg": "#2a78d6", "2. Rank avg": "#eb6834",
          "3. Weighted logit": "#1baf7a", "4. LR stacking": "#eda100"}

## 0. Combine `fusion_df`

In [2]:
cnn_df = pd.read_csv("validation_predictions.csv")
cnn_df

FileNotFoundError: [Errno 2] No such file or directory: 'validation_predictions.csv'

In [ ]:
REQUIRED = ["image_name", "patient_id", "target", "p_cnn", "p_rf"]
assert set(REQUIRED) <= set(fusion_df.columns), f"fusion_df needs columns {REQUIRED}"
assert fusion_df[REQUIRED].notna().all().all(), "missing values in fusion_df"
assert fusion_df["image_name"].is_unique, "duplicate images in fusion_df"
fusion_df = fusion_df.reset_index(drop=True)

print(f"Validation rows: {len(fusion_df)} | melanomas: {fusion_df['target'].sum()} "
      f"| patients: {fusion_df['patient_id'].nunique()}")
fusion_df[["p_cnn", "p_rf"]].describe().T

### Correlation Check
The two models score on different scales. A CNN trained with class weights pushes benign cases to
~0.1–0.2, not ~0.02, and the RF has its own scale. That is why methods 2–4 do not average the raw
probabilities. The correlation shows how much the models **disagree**. Low correlation means more to
gain from fusion.

In [ ]:
print("Correlation (Spearman) between p_cnn and p_rf:",
      round(fusion_df[["p_cnn", "p_rf"]].corr(method="spearman").iloc[0, 1], 3))

fig, ax = plt.subplots(figsize=(5.5, 5))
for t, c, lab, s in [(0, "#c3c2b7", "benign", 10), (1, "#e34948", "melanoma", 40)]:
    d = fusion_df[fusion_df["target"] == t]
    ax.scatter(d["p_cnn"], d["p_rf"], s=s, c=c, label=lab, edgecolors="white", linewidths=0.5)
ax.set_xlabel("p_cnn"); ax.set_ylabel("p_rf"); ax.set_title("Validation: CNN vs RF probability")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 1. The four fusion models
Every method has the same `fit(df)` / `predict(df)` interface, so the CV loop and the test evaluation
can treat them identically. `predict` returns a **score**: higher means more likely melanoma. Only
method 4 returns a calibrated probability.

In [ ]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), EPS, 1 - EPS)
    return np.log(p / (1 - p))


class BaseOnly:
    # Reference point: one base model's own probability, no fusion.
    def __init__(self, col): self.col = col
    def fit(self, df): return self
    def predict(self, df): return df[self.col].to_numpy(dtype=float)


class SimpleAverage:
    # Method 1: (p_cnn + p_rf) / 2. Nothing to learn.
    def fit(self, df): return self
    def predict(self, df):
        return (df["p_cnn"].to_numpy(float) + df["p_rf"].to_numpy(float)) / 2


class RankAverage:
    # Method 2: average of percentiles.
    # fit() stores the training score distribution of each model. predict() maps a new score to
    # "fraction of training scores below it" (an empirical CDF). New data is never ranked against
    # itself, so test scores do not depend on the rest of the test set, and a threshold chosen on
    # validation carries over.
    def fit(self, df):
        self.ref = {c: np.sort(df[c].to_numpy(float)) for c in ["p_cnn", "p_rf"]}
        return self
    def _pct(self, col, x):
        r = self.ref[col]
        # midpoint of left/right insertion -> ties get the average rank
        return (np.searchsorted(r, x, "left") + np.searchsorted(r, x, "right")) / (2 * len(r))
    def predict(self, df):
        return (self._pct("p_cnn", df["p_cnn"].to_numpy(float)) +
                self._pct("p_rf",  df["p_rf"].to_numpy(float))) / 2


class WeightedLogit:
    # Method 3: w * logit(p_cnn) + (1 - w) * logit(p_rf), with w chosen on the training data.
    # w = 1 -> CNN only, w = 0 -> RF only. The chosen w is the "how much do we trust the image" number.
    # w is picked by ROC-AUC because AUC is smooth and stable with very few positives.
    # (PR-AUC on ~20 positives jumps around too much to tune on.)
    def __init__(self, grid=np.round(np.linspace(0, 1, 21), 2)): self.grid = grid
    def _score(self, df, w):
        return w * logit(df["p_cnn"]) + (1 - w) * logit(df["p_rf"])
    def fit(self, df):
        y = df["target"].to_numpy()
        self.curve_ = np.array([roc_auc_score(y, self._score(df, w)) for w in self.grid])
        best = np.flatnonzero(self.curve_ >= self.curve_.max() - 1e-12)
        self.w_ = float(self.grid[best].mean())   # middle of any tie plateau, not an arbitrary edge
        return self
    def predict(self, df): return self._score(df, self.w_)


class LRStacking:
    # Method 4: logistic regression meta-learner on the two logits.
    #   P(melanoma) = sigmoid(a + b * logit(p_cnn) + c * logit(p_rf))
    # No class_weight, so the output stays a calibrated probability at the real prevalence.
    # L2 penalty (C=1) shrinks b and c a little, which keeps them stable with ~25 positives.
    def __init__(self, C=1.0): self.C = C
    def _X(self, df): return np.column_stack([logit(df["p_cnn"]), logit(df["p_rf"])])
    def fit(self, df):
        self.model_ = LogisticRegression(C=self.C, max_iter=1000).fit(self._X(df), df["target"])
        return self
    def predict(self, df): return self.model_.predict_proba(self._X(df))[:, 1]
    @property
    def coefs_(self):
        return dict(intercept=self.model_.intercept_[0],
                    b_cnn=self.model_.coef_[0, 0], c_rf=self.model_.coef_[0, 1])


def make_models():
    # Fresh, unfitted instances, so no state leaks between CV folds.
    return {"CNN only": BaseOnly("p_cnn"), "RF only": BaseOnly("p_rf"),
            "1. Simple avg": SimpleAverage(), "2. Rank avg": RankAverage(),
            "3. Weighted logit": WeightedLogit(), "4. LR stacking": LRStacking()}

FUSION_NAMES = ["1. Simple avg", "2. Rank avg", "3. Weighted logit", "4. LR stacking"]

## 2. Metric helpers
* **ROC-AUC**: chance that a random melanoma scores higher than a random benign lesion. It ignores prevalence.
* **PR-AUC (average precision)**: how clean the top of the ranking is. This is the more honest metric at
  ~2% prevalence, where random guessing scores ≈ 0.02 (not 0.5).
* **Sensitivity / specificity** at a threshold chosen on *training* data to reach `TARGET_SENS`.
* **Patient-level bootstrap**: resample *patients* (not images) with replacement and recompute the
  metrics. This gives confidence intervals that reflect how few melanomas and patients there are.
  Every method is scored on the same resamples, so differences between methods are paired.

In [ ]:
def threshold_for_sensitivity(y, s, target=TARGET_SENS):
    # Highest threshold whose sensitivity on (y, s) is >= target.
    # A higher threshold means fewer false positives, so take the highest one that still catches
    # enough melanomas.
    pos = np.sort(s[y == 1])[::-1]                 # positive scores, high -> low
    k = int(np.ceil(target * len(pos)))            # need at least k positives at or above threshold
    return pos[k - 1]

def patient_bootstrap(df, scores, n_boot=N_BOOT, seed=SEED):
    # Resample patients with replacement and recompute AUCs for every method on the same resamples.
    # Returns a dict of arrays: model -> (n_boot, 2) array of [roc_auc, pr_auc].
    rng = np.random.default_rng(seed)
    pids = df["patient_id"].to_numpy()
    uniq = np.unique(pids)
    rows_of = {p: np.flatnonzero(pids == p) for p in uniq}
    y = df["target"].to_numpy()
    out = {m: [] for m in scores}
    for _ in range(n_boot):
        idx = np.concatenate([rows_of[p] for p in rng.choice(uniq, len(uniq), replace=True)])
        if y[idx].min() == y[idx].max():
            continue                       # resample without both classes -> AUC undefined
        for m, s in scores.items():
            out[m].append((roc_auc_score(y[idx], s[idx]), average_precision_score(y[idx], s[idx])))
    return {m: np.array(v) for m, v in out.items()}


def threshold_metrics(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(sens=tp / (tp + fn), spec=tn / (tn + fp),
                ppv=tp / (tp + fp) if tp + fp else np.nan, tp=tp, fp=fp, fn=fn, tn=tn)

## 3. Repeated patient-grouped cross-validation on the validation set
`StratifiedGroupKFold` keeps every image of a patient in the same fold, which stops leakage through
look-alike lesions on one person. It also spreads the melanomas evenly across folds.

Repeating the CV with different shuffles measures how much the *fitted* methods (3, 4) depend on the
split. It says nothing about sampling uncertainty: methods 1 and 2 give the same OOF scores on every
repeat, so their SD across repeats is ~0. Section 3a therefore also bootstraps patients on the OOF
predictions to get real confidence intervals.

Within each repeat, OOF predictions from the 5 folds are **pooled** before computing AUCs. With only
~5 positives per fold, per-fold AUCs would be very noisy. Pooling is safe for methods 1, 2 and 4.
For method 3, the score scale shifts slightly when $w$ differs between folds, so its pooled AUC is a
little pessimistic, if anything.

In [ ]:
y_all, groups = fusion_df["target"].to_numpy(), fusion_df["patient_id"].to_numpy()
cv_rows, w_per_fold, coef_per_fold = [], [], []
oof_first = None    # OOF scores of repeat 0, kept for plotting curves

for rep in range(N_REPEATS):
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED + rep)
    oof = {name: np.zeros(len(fusion_df)) for name in make_models()}
    oof_pred = {name: np.zeros(len(fusion_df), dtype=int) for name in make_models()}

    for tr_idx, va_idx in sgkf.split(fusion_df, y_all, groups):
        tr, va = fusion_df.iloc[tr_idx], fusion_df.iloc[va_idx]
        for name, model in make_models().items():
            model.fit(tr)
            thr = threshold_for_sensitivity(tr["target"].to_numpy(), model.predict(tr))
            s_va = model.predict(va)
            oof[name][va_idx] = s_va
            oof_pred[name][va_idx] = (s_va >= thr).astype(int)
            if name == "3. Weighted logit": w_per_fold.append(model.w_)
            if name == "4. LR stacking":    coef_per_fold.append(model.coefs_)

    for name in oof:
        cv_rows.append(dict(repeat=rep, model=name,
                            roc_auc=roc_auc_score(y_all, oof[name]),
                            pr_auc=average_precision_score(y_all, oof[name]),
                            **threshold_metrics(y_all, oof_pred[name])))
    if rep == 0:
        oof_first = oof

cv_df = pd.DataFrame(cv_rows)
print(f"Done: {N_REPEATS} repeats x {N_SPLITS} folds")

### 3a. Cross-validated results
`mean` / `std` = across CV repeats (split variability). `ci95` = patient-bootstrap 95% CI of the OOF
metric (sampling uncertainty, the one to report). `P(beats CNN)` = share of bootstrap resamples where
the method beats CNN-only.

In [ ]:
order = list(make_models())
cv_summary = (cv_df.groupby("model")[["roc_auc", "pr_auc", "sens", "spec", "ppv"]]
              .agg(["mean", "std"]).reindex(order))

# Uncertainty: patient-level bootstrap of the OOF predictions (CV repeat 0).
# Every method is scored on the same resamples, so "P(beats CNN)" is a paired comparison.
val_boot = patient_bootstrap(fusion_df, oof_first)
for metric, j in [("roc_auc", 0), ("pr_auc", 1)]:
    cv_summary[(metric, "ci95")] = [f"[{np.percentile(val_boot[m][:, j], 2.5):.3f}, "
                                    f"{np.percentile(val_boot[m][:, j], 97.5):.3f}]" for m in order]
    cv_summary[("P(beats CNN)", metric)] = [(val_boot[m][:, j] > val_boot["CNN only"][:, j]).mean()
                                            for m in order]
cv_summary = cv_summary[["roc_auc", "pr_auc", "sens", "spec", "ppv", "P(beats CNN)"]]
cv_summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, metric, title in [(axes[0], "roc_auc", "ROC-AUC"), (axes[1], "pr_auc", "PR-AUC")]:
    m = cv_summary[(metric, "mean")]
    j = 0 if metric == "roc_auc" else 1
    lo = np.array([np.percentile(val_boot[k][:, j], 2.5) for k in order[::-1]])
    hi = np.array([np.percentile(val_boot[k][:, j], 97.5) for k in order[::-1]])
    mv = m[order[::-1]].to_numpy()
    ax.barh(order[::-1], mv, xerr=[np.clip(mv - lo, 0, None), np.clip(hi - mv, 0, None)], height=0.6,
            color=[COLORS[k] for k in order[::-1]], error_kw=dict(ecolor="#52514e", lw=1, capsize=3))
    for i, k in enumerate(order[::-1]):
        ax.text(max(hi[i], m[k]) + 0.01, i, f"{m[k]:.3f}", va="center", fontsize=9, color="#0b0b0b")
    if metric == "pr_auc":
        prev = y_all.mean()
        ax.axvline(prev, color="#52514e", ls=":", lw=1)
        ax.text(prev, -0.8, f" chance = {prev:.3f}", fontsize=8, color="#52514e")
    ax.set_title(f"CV {title} (OOF, patient-bootstrap 95% CI)")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="x", color="#e6e5e0", lw=0.8)
    ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

### 3b. What did the learned methods learn?
* **Method 3**: the distribution of $w$ across folds shows how much the image is trusted relative to
  metadata. If it is stable around one value, the trade-off is real. If it is spread from 0 to 1, the
  data cannot tell.
* **Method 4**: $b$ and $c$ are the log-odds weights on each model. Because both inputs are logits,
  $b / (b + c)$ is comparable to $w$.

In [ ]:
w_arr = np.array(w_per_fold)
coef_df = pd.DataFrame(coef_per_fold)
coef_df["cnn_share"] = coef_df["b_cnn"] / (coef_df["b_cnn"] + coef_df["c_rf"])

print(f"Method 3, w (CNN weight) across {len(w_arr)} folds: "
      f"median {np.median(w_arr):.2f}, IQR [{np.percentile(w_arr,25):.2f}, {np.percentile(w_arr,75):.2f}]")
display(coef_df.describe().loc[["mean", "std", "min", "max"]].round(3))

# AUC as a function of w on the FULL validation set: the shape of the trade-off
wl_full = WeightedLogit().fit(fusion_df)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(wl_full.grid, wl_full.curve_, color=COLORS["3. Weighted logit"], lw=2, marker="o", ms=4)
ax.axvline(wl_full.w_, color="#52514e", ls=":", lw=1)
ax.text(wl_full.w_, ax.get_ylim()[0], f"  best w = {wl_full.w_:.2f}", fontsize=9, color="#52514e", va="bottom")
ax.set_xlabel("w   (0 = RF only, 1 = CNN only)"); ax.set_ylabel("ROC-AUC on validation")
ax.set_title("Method 3: how the weight on the image changes AUC")
ax.spines[["top", "right"]].set_visible(False); ax.grid(color="#e6e5e0", lw=0.8)
plt.tight_layout(); plt.show()

### 3c. ROC and precision-recall curves (OOF predictions, CV repeat 0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name in order:
    s = oof_first[name]
    base = name in ("CNN only", "RF only")
    kw = dict(color=COLORS[name], lw=1.5 if base else 2, ls="--" if base else "-")
    fpr, tpr, _ = roc_curve(y_all, s)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_all, s):.3f})", **kw)
    prec, rec, _ = precision_recall_curve(y_all, s)
    axes[1].plot(rec, prec, label=f"{name} ({average_precision_score(y_all, s):.3f})", **kw)
axes[0].plot([0, 1], [0, 1], color="#e6e5e0", lw=1)
axes[0].set(xlabel="False positive rate (1 - specificity)", ylabel="Sensitivity", title="ROC (OOF)")
axes[1].axhline(y_all.mean(), color="#e6e5e0", lw=1)
axes[1].set(xlabel="Recall (sensitivity)", ylabel="Precision", title="Precision-recall (OOF)")
for ax in axes:
    ax.legend(frameon=False, fontsize=8, loc="lower right" if ax is axes[0] else "upper right")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 4. Pick the winner on validation
Primary metric: **CV PR-AUC** (the clinically relevant ranking quality at 2% prevalence).
Tie-breaker: CV ROC-AUC.

**Simplicity rule (one-standard-error rule):** go from the simplest method (1) to the most complex (4).
Keep the first method whose PR-AUC is within one bootstrap standard error of the top scorer's, using the
paired patient-bootstrap difference. With 25 positives, a gain smaller than the noise is not a gain,
and a simpler method is cheaper to explain and less likely to overfit.

In [ ]:
fusion_cv = cv_summary.loc[FUSION_NAMES]
ranked = fusion_cv.sort_values([("pr_auc", "mean"), ("roc_auc", "mean")], ascending=False)
best_raw = ranked.index[0]

# Gap to the top scorer, in units of the paired bootstrap SE
gap = {}
for m in FUSION_NAMES:
    d = val_boot[best_raw][:, 1] - val_boot[m][:, 1]          # PR-AUC difference on the same resamples
    gap[m] = (cv_summary.loc[best_raw, ("pr_auc", "mean")] - cv_summary.loc[m, ("pr_auc", "mean")],
              d.std(ddof=1))
chosen = next(m for m in FUSION_NAMES if gap[m][0] <= gap[m][1])   # FUSION_NAMES is simple -> complex

print("PR-AUC gap to the top scorer (difference, paired bootstrap SE):")
for m in FUSION_NAMES:
    print(f"  {m:<18} {gap[m][0]:.3f}  (SE {gap[m][1]:.3f})")

print("Ranking by CV PR-AUC:")
display(ranked[[("pr_auc", "mean"), ("pr_auc", "ci95"), ("roc_auc", "mean"), ("roc_auc", "ci95"),
                ("sens", "mean"), ("spec", "mean")]].round(3))
print(f"Highest CV PR-AUC : {best_raw}")
print(f"Chosen (simplicity rule) : {chosen}")

## 5. Fit on the full validation set (final models)
Each method is refit on **all** of `fusion_df`. The operating threshold for each method comes from the
**pooled OOF scores** of CV repeat 0, not from the in-sample fit. This matters most for method 4: its
in-sample probabilities are slightly over-confident, which would pick a threshold that misses more
melanomas on new data.

*(For methods 1–2, OOF and in-sample scores are identical apart from method 2's reference
distribution, so this makes little difference. For methods 3–4, their scale changes slightly between
folds and the full fit, so the threshold is approximate. That is acceptable, because the AUCs do not
depend on it.)*

In [ ]:
final_models = {name: model.fit(fusion_df) for name, model in make_models().items()}
final_thr = {name: threshold_for_sensitivity(y_all, final_models[name].predict(fusion_df))
             if name in ("CNN only", "RF only", "1. Simple avg", "2. Rank avg")
             else threshold_for_sensitivity(y_all, oof_first[name])
             for name in final_models}

print(f"Method 3 final w = {final_models['3. Weighted logit'].w_:.2f}")
print("Method 4 final coefficients:",
      {k: round(v, 3) for k, v in final_models['4. LR stacking'].coefs_.items()})
pd.Series(final_thr, name=f"threshold for sens >= {TARGET_SENS}").round(4)

## 6. Test set: run **once**, after the winner is fixed
Requires `fusion_test_df` with the same columns. The test predictions **must** come from the same
CNN and RF that produced the validation predictions (trained on the 5,630-image training split).
If the base models are retrained on train + validation, their score scales shift and the fusion
weights no longer apply.

Reported for every method: ROC-AUC and PR-AUC with **patient-level bootstrap** 95% CIs (resampling
patients, not images, because images from one patient are correlated), sensitivity, specificity and
PPV at the validation threshold, and a **paired bootstrap of ΔAUC vs CNN only**. If the ΔAUC CI
excludes 0, fusion beats the image alone.

In [ ]:
if "fusion_test_df" in globals():
    assert set(REQUIRED) <= set(fusion_test_df.columns), f"fusion_test_df needs columns {REQUIRED}"
    test = fusion_test_df.reset_index(drop=True)
    y_te = test["target"].to_numpy()
    te_scores = {name: m.predict(test) for name, m in final_models.items()}
    boot = patient_bootstrap(test, te_scores)

    rows = []
    for name, s in te_scores.items():
        b = boot[name]
        d = b - boot["CNN only"]          # paired: same resamples
        rows.append(dict(
            model=name,
            roc_auc=roc_auc_score(y_te, s),
            roc_ci=f"[{np.percentile(b[:,0],2.5):.3f}, {np.percentile(b[:,0],97.5):.3f}]",
            pr_auc=average_precision_score(y_te, s),
            pr_ci=f"[{np.percentile(b[:,1],2.5):.3f}, {np.percentile(b[:,1],97.5):.3f}]",
            d_roc_vs_cnn_ci=f"[{np.percentile(d[:,0],2.5):+.3f}, {np.percentile(d[:,0],97.5):+.3f}]",
            d_pr_vs_cnn_ci=f"[{np.percentile(d[:,1],2.5):+.3f}, {np.percentile(d[:,1],97.5):+.3f}]",
            **threshold_metrics(y_te, (s >= final_thr[name]).astype(int))))
    test_results = pd.DataFrame(rows).set_index("model")
    print(f"Test rows: {len(test)} | melanomas: {y_te.sum()} | patients: {test['patient_id'].nunique()}")
    print(f"Pre-registered winner (chosen on validation): {chosen}")
    display(test_results.round(3))
else:
    print("fusion_test_df not defined yet: skipping test evaluation.")

In [ ]:
if "fusion_test_df" in globals():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw=dict(width_ratios=[1, 1, 0.8]))
    for name in order:
        s = te_scores[name]
        base = name in ("CNN only", "RF only")
        kw = dict(color=COLORS[name], lw=1.5 if base else 2, ls="--" if base else "-")
        fpr, tpr, _ = roc_curve(y_te, s)
        axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_te, s):.3f})", **kw)
        prec, rec, _ = precision_recall_curve(y_te, s)
        axes[1].plot(rec, prec, label=f"{name} ({average_precision_score(y_te, s):.3f})", **kw)
    axes[0].plot([0, 1], [0, 1], color="#e6e5e0", lw=1)
    axes[0].set(xlabel="False positive rate", ylabel="Sensitivity", title="Test ROC")
    axes[1].axhline(y_te.mean(), color="#e6e5e0", lw=1)
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Test precision-recall")
    axes[0].legend(frameon=False, fontsize=8, loc="lower right")
    axes[1].legend(frameon=False, fontsize=8, loc="upper right")

    cm = confusion_matrix(y_te, (te_scores[chosen] >= final_thr[chosen]).astype(int))
    axes[2].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        axes[2].text(j, i, f"{v}", ha="center", va="center", fontsize=13,
                     color="white" if v > cm.max() / 2 else "#0b0b0b")
    axes[2].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["benign", "melanoma"],
                yticklabels=["benign", "melanoma"], xlabel="Predicted", ylabel="Actual",
                title=f"Test confusion matrix: {chosen}")
    for ax in axes[:2]:
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

## 7. Final comparison: validation CV vs test
One table to put in the report. Use the **validation CV** columns to justify the choice of fusion
method, and the **test** columns to report how well it generalises. Read the test columns with their
confidence intervals: on ~35 test melanomas, differences of a few points of AUC are usually inside
the noise.

In [ ]:
final_table = pd.DataFrame({
    "CV ROC-AUC": cv_summary[("roc_auc", "mean")].round(3),
    "CV ROC 95% CI": cv_summary[("roc_auc", "ci95")],
    "CV PR-AUC":  cv_summary[("pr_auc", "mean")].round(3),
    "CV PR 95% CI": cv_summary[("pr_auc", "ci95")],
    "CV sens":    cv_summary[("sens", "mean")].round(3),
    "CV spec":    cv_summary[("spec", "mean")].round(3),
    "CV P(PR-AUC > CNN)": cv_summary[("P(beats CNN)", "pr_auc")].round(2),
})
if "fusion_test_df" in globals():
    final_table = final_table.join(test_results[["roc_auc", "roc_ci", "pr_auc", "pr_ci",
                                                 "d_pr_vs_cnn_ci", "sens", "spec"]]
                                   .round(3).add_prefix("Test "))
final_table["chosen"] = ["<-- winner" if m == chosen else "" for m in final_table.index]
final_table